# Gwas for new potato

This notebook performs a genome-wide association study (GWAS) using the GEMMA_taglotype pipeline.

In [0]:
# Import the GEMMA_taglotype class from the shared repository
import sys
import pandas as pd
sys.path.append("/Workspace/Shared_Repos")
from gwas_analysis.gemma_taglotype import GEMMA_taglotype

In [0]:
import os
import yaml
import pandas as pd

CONFIG_PATH = "/Volumes/bmqg/default_bronze/fatemeh/config_mixed.yaml"

with open(CONFIG_PATH, "r") as f:
    CONFIG = yaml.safe_load(f)

PHENO_PATH = CONFIG["paths"]["aroma_matrix_newharvested"]
assert os.path.exists(PHENO_PATH), f"Phenotype file not found: {PHENO_PATH}"

# 1) try common separators and pick the one that yields the most columns
seps = [",", ";", "\t"]
best_df, best_sep = None, None

for sep in seps:
    try:
        tmp = pd.read_csv(PHENO_PATH, sep=sep, engine="python")
        if best_df is None or tmp.shape[1] > best_df.shape[1]:
            best_df, best_sep = tmp, sep
    except Exception:
        pass

assert best_df is not None, "Could not read phenotype file with common separators"
df_pheno = best_df
print("Detected separator:", repr(best_sep))
print("Raw shape:", df_pheno.shape)

# 2) if it still came in as 1 column, split that column by comma (your header shows comma-separated)
if df_pheno.shape[1] == 1 and "Variety" not in df_pheno.columns:
    only_col = df_pheno.columns[0]
    # re-read as a single-column file, then split
    df_one = pd.read_csv(PHENO_PATH, header=None, names=["_raw"], engine="python")
    split = df_one["_raw"].str.split(",", expand=True)
    split.columns = split.iloc[0].tolist()
    df_pheno = split.iloc[1:].copy()
    print("Recovered shape after split:", df_pheno.shape)

assert "Variety" in df_pheno.columns, f"Column 'Variety' not found. Columns: {df_pheno.columns[:5]}"

# 3) clean Variety
df_pheno["Variety"] = df_pheno["Variety"].astype(str).str.strip().str.upper()

# 4) convert numeric columns properly (important!)
for c in df_pheno.columns:
    if c != "Variety":
        df_pheno[c] = pd.to_numeric(df_pheno[c], errors="coerce")

# 5) aggregate
df_pheno = (
    df_pheno
    .groupby("Variety", as_index=True)
    .mean(numeric_only=True)
    .sort_index()
)

print("Shape:", df_pheno.shape)
print("Index name:", df_pheno.index.name)
print("Index unique:", df_pheno.index.is_unique)
print("First index values:", df_pheno.index[:10].tolist())

display(df_pheno.head())

In [0]:
import shutil, os
shutil.rmtree("/tmp/gemma_local", ignore_errors=True)

local_tmp = "/tmp/gemma_local"
os.makedirs(local_tmp, exist_ok=True)

print(local_tmp)


In [0]:
run_path = "/tmp/tmp_gemma/run_20260305"

os.makedirs(run_path, exist_ok=True)
print("Run path ready:", run_path)


In [0]:
print("Phenotype n:", df_pheno.shape[0])
print("Phenotype IDs sample:", df_pheno.index[:10].tolist())

In [0]:
gemma = GEMMA_taglotype(
    df_pheno,
    min_allele_freq=0.007,
    max_allele_freq=0.8,
    max_bad_fraction=0.1,
    max_combinations=4,
    ploidy=4,
    run_id="run_local_20260305_newharvested"
)

gemma.run_gwas()
